In [ ]:
#Setup
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time
from torch.utils.data import TensorDataset, DataLoader

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
train_path = "/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv"
test_path = "/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

pixel_columns = [col for col in train_df.columns if col.startswith("pixel")]
assert len(pixel_columns) == 784, "Expected 784 pixel columns"
print("Number of pixel columns verified:", len(pixel_columns))

X_full_train = train_df[pixel_columns].values.astype(np.float32) / 255.0
y_full_train = train_df["label"].values.astype(np.int64)

X_test = test_df[pixel_columns].values.astype(np.float32) / 255.0
y_test = test_df["label"].values.astype(np.int64)

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_full_train, y_full_train, test_size=0.2, random_state=SEED, stratify=y_full_train
)

print("Total training samples (before split):", X_full_train.shape[0])
print("Training samples after split:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Test samples:", X_test.shape[0])
print("Input shape:", X_train.shape[1])
print("Number of classes:", len(np.unique(y_full_train)))

class_names = {
    0: "T-shirt/top", 1: "Trouser", 2: "Pullover", 3: "Dress", 4: "Coat",
    5: "Sandal", 6: "Shirt", 7: "Sneaker", 8: "Bag", 9: "Ankle boot"
}

def class_distribution_table(y, name):
    counts = pd.Series(y).value_counts().sort_index()
    table = pd.DataFrame({
        "Class": counts.index,
        "Class Name": [class_names[i] for i in counts.index],
        "Count": counts.values
    })
    print("\nClass distribution -", name)
    print(table.to_string(index=False))

class_distribution_table(y_train, "Training set")
class_distribution_table(y_val, "Validation set")
class_distribution_table(y_test, "Test set")

In [ ]:
def initialize_parameters(input_size, hidden_size, output_size, seed):
    rng = np.random.RandomState(seed)
    W1 = rng.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
    b1 = np.zeros((1, hidden_size))
    W2 = rng.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
    b2 = np.zeros((1, output_size))
    return W1, b1, W2, b2

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(np.float32)

def softmax(Z):
    Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

def one_hot_encode(y, num_classes):
    one_hot = np.zeros((y.shape[0], num_classes))
    one_hot[np.arange(y.shape[0]), y] = 1
    return one_hot

def forward_pass(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = softmax(Z2)
    cache = (X, Z1, A1, Z2, A2)
    return A2, cache

def cross_entropy_loss(A2, y_one_hot):
    epsilon = 1e-8
    N = A2.shape[0]
    loss = -np.sum(y_one_hot * np.log(A2 + epsilon)) / N
    return loss

def backward_pass(cache, y_one_hot, W2):
    X, Z1, A1, Z2, A2 = cache
    N = X.shape[0]
    dZ2 = (A2 - y_one_hot) / N
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    return dW1, db1, dW2, db2

def update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate):
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    return W1, b1, W2, b2

In [ ]:
subset_size = 5000
X_subset = X_train[:subset_size]
y_subset = y_train[:subset_size]
y_subset_one_hot = one_hot_encode(y_subset, 10)

hidden_size = 64
input_size = 784
output_size = 10
num_epochs = 20
batch_size = 64
learning_rate = 0.1

print("Batch size:", batch_size)
print("Learning rate:", learning_rate)

W1, b1, W2, b2 = initialize_parameters(input_size, hidden_size, output_size, SEED)

training_losses = []
num_samples = X_subset.shape[0]

for epoch in range(num_epochs):
    rng = np.random.RandomState(SEED + epoch)
    permutation = rng.permutation(num_samples)
    X_shuffled = X_subset[permutation]
    y_shuffled = y_subset_one_hot[permutation]

    epoch_loss = 0.0
    num_batches = 0

    for start in range(0, num_samples, batch_size):
        end = start + batch_size
        X_batch = X_shuffled[start:end]
        y_batch = y_shuffled[start:end]

        A2, cache = forward_pass(X_batch, W1, b1, W2, b2)
        loss = cross_entropy_loss(A2, y_batch)
        dW1, db1, dW2, db2 = backward_pass(cache, y_batch, W2)
        W1, b1, W2, b2 = update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)

        epoch_loss += loss
        num_batches += 1

    avg_epoch_loss = epoch_loss / num_batches
    training_losses.append(avg_epoch_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {avg_epoch_loss:.4f}")

print("Final training loss:", training_losses[-1])

plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), training_losses, label="Training Loss")
plt.title("Part 1: NumPy MLP Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.show()

In [ ]:
W1_check, b1_check, W2_check, b2_check = initialize_parameters(input_size, hidden_size, output_size, SEED)

X_check = X_train[:batch_size]
y_check = y_train[:batch_size]
y_check_one_hot = one_hot_encode(y_check, 10)

A2_check, cache_check = forward_pass(X_check, W1_check, b1_check, W2_check, b2_check)
dW1_np, db1_np, dW2_np, db2_np = backward_pass(cache_check, y_check_one_hot, W2_check)

class TorchMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(TorchMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        z1 = self.fc1(x)
        a1 = torch.relu(z1)
        z2 = self.fc2(a1)
        return z2

torch_model = TorchMLP(input_size, hidden_size, output_size)

with torch.no_grad():
    torch_model.fc1.weight.copy_(torch.tensor(W1_check.T, dtype=torch.float32))
    torch_model.fc1.bias.copy_(torch.tensor(b1_check.flatten(), dtype=torch.float32))
    torch_model.fc2.weight.copy_(torch.tensor(W2_check.T, dtype=torch.float32))
    torch_model.fc2.bias.copy_(torch.tensor(b2_check.flatten(), dtype=torch.float32))

X_check_tensor = torch.tensor(X_check, dtype=torch.float32)
y_check_tensor = torch.tensor(y_check, dtype=torch.long)

logits = torch_model(X_check_tensor)
loss_fn = nn.CrossEntropyLoss()
loss_check_torch = loss_fn(logits, y_check_tensor)
loss_check_torch.backward()

dW1_torch = torch_model.fc1.weight.grad.detach().numpy().T
db1_torch = torch_model.fc1.bias.grad.detach().numpy().reshape(1, -1)
dW2_torch = torch_model.fc2.weight.grad.detach().numpy().T
db2_torch = torch_model.fc2.bias.grad.detach().numpy().reshape(1, -1)

W1_diff = np.max(np.abs(dW1_np - dW1_torch))
b1_diff = np.max(np.abs(db1_np - db1_torch))
W2_diff = np.max(np.abs(dW2_np - dW2_torch))
b2_diff = np.max(np.abs(db2_np - db2_torch))

print("W1 max absolute gradient difference:", W1_diff)
print("b1 max absolute gradient difference:", b1_diff)
print("W2 max absolute gradient difference:", W2_diff)
print("b2 max absolute gradient difference:", b2_diff)

overall_max_diff = max(W1_diff, b1_diff, W2_diff, b2_diff)
print("Overall maximum absolute difference:", overall_max_diff)

If the overall maximum absolute difference is on the order of 1e-6 or smaller (this is expected float32 precision noise), the manual NumPy backpropagation matches PyTorch's autograd and can be considered correct.

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, activation_name):
        super(BaselineMLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.activation_name = activation_name

    def apply_activation(self, x):
        if self.activation_name == "sigmoid":
            return torch.sigmoid(x)
        elif self.activation_name == "tanh":
            return torch.tanh(x)
        elif self.activation_name == "relu":
            return torch.relu(x)
        elif self.activation_name == "leaky_relu":
            return F.leaky_relu(x, negative_slope=0.01)

    def forward(self, x):
        a1 = self.apply_activation(self.fc1(x))
        a2 = self.apply_activation(self.fc2(a1))
        logits = self.fc3(a2)
        return logits

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

BATCH_SIZE = 128
LEARNING_RATE = 0.01
NUM_EPOCHS = 15

def evaluate_model(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        loss = nn.CrossEntropyLoss()(logits, y).item()
        predictions = torch.argmax(logits, dim=1)
        accuracy = (predictions == y).float().mean().item()
    return loss, accuracy

def train_activation_model(activation_name):
    set_seed(SEED)
    model = BaselineMLP(activation_name).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()

    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

    train_losses, val_losses, val_accuracies = [], [], []
    first_epoch_grad, final_epoch_grad = None, None

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss, num_batches, epoch_grad_sum = 0.0, 0, 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()

            grad_mean = model.fc1.weight.grad.abs().mean().item()
            epoch_grad_sum += grad_mean

            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1

        avg_train_loss = epoch_loss / num_batches
        avg_grad = epoch_grad_sum / num_batches
        train_losses.append(avg_train_loss)

        val_loss, val_accuracy = evaluate_model(model, X_val_tensor, y_val_tensor)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)

        if epoch == 0:
            first_epoch_grad = avg_grad
        if epoch == NUM_EPOCHS - 1:
            final_epoch_grad = avg_grad

        print(f"[{activation_name}] Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {avg_train_loss:.4f} - Val Loss: {val_loss:.4f} - Val Accuracy: {val_accuracy:.4f}")

    return {
        "model": model, "train_losses": train_losses, "val_losses": val_losses,
        "val_accuracies": val_accuracies, "first_epoch_grad": first_epoch_grad,
        "final_epoch_grad": final_epoch_grad
    }

activation_names = ["sigmoid", "tanh", "relu", "leaky_relu"]
activation_results = {}

for name in activation_names:
    print("\nTraining model with activation:", name)
    activation_results[name] = train_activation_model(name)

Sigmoid squashes its input into (0,1) and its derivative maxes out at 0.25, so gradients shrink every time they pass through it during backpropagation — across two hidden layers this compounds and the first layer's gradient becomes very small (vanishing gradient). ReLU's derivative is either 0 or exactly 1 for active units, so it doesn't shrink the gradient the same way, which is why its first-layer gradient stays comparatively larger across epochs.

In [ ]:
relu_model = activation_results["relu"]["model"]
relu_model.eval()

val_batch_size = 256
X_val_batch = X_val_tensor[:val_batch_size]

with torch.no_grad():
    a1 = torch.relu(relu_model.fc1(X_val_batch))

dead_units = (a1.sum(dim=0) == 0).sum().item()
total_units = a1.shape[1]
dead_percentage = (dead_units / total_units) * 100

print("Dead hidden units:", dead_units)
print("Total hidden units:", total_units)
print("Dead ReLU percentage:", dead_percentage)

A unit counted as "dead" here fired zero for every single sample in this validation batch — meaning it contributes nothing to the output and passes no gradient backward for these inputs. A high percentage suggests wasted network capacity; a low percentage means most units are still doing useful work

In [ ]:
class ClassificationMLP(nn.Module):
    def __init__(self):
        super(ClassificationMLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        a1 = torch.relu(self.fc1(x))
        a2 = torch.relu(self.fc2(a1))
        return self.fc3(a2)

def train_loss_function_model(loss_type):
    set_seed(SEED)
    model = ClassificationMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

    train_losses = []

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss, num_batches = 0.0, 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(X_batch)

            if loss_type == "cross_entropy":
                loss = nn.CrossEntropyLoss()(logits, y_batch)
            elif loss_type == "mse":
                probabilities = torch.softmax(logits, dim=1)
                y_one_hot = F.one_hot(y_batch, num_classes=10).float()
                loss = nn.MSELoss()(probabilities, y_one_hot)

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1

        avg_train_loss = epoch_loss / num_batches
        train_losses.append(avg_train_loss)
        print(f"[{loss_type}] Epoch {epoch+1}/{NUM_EPOCHS} - Training Loss: {avg_train_loss:.4f}")

    return model, train_losses

cce_model, cce_train_losses = train_loss_function_model("cross_entropy")
mse_model, mse_train_losses = train_loss_function_model("mse")

def compute_test_accuracy(model):
    model.eval()
    with torch.no_grad():
        logits = model(X_test_tensor)
        predictions = torch.argmax(logits, dim=1)
        return (predictions == y_test_tensor).float().mean().item()

cce_test_accuracy = compute_test_accuracy(cce_model)
mse_test_accuracy = compute_test_accuracy(mse_model)

loss_function_table = pd.DataFrame({
    "Loss Function": ["Categorical Cross-Entropy", "Mean Squared Error"],
    "Test Accuracy": [cce_test_accuracy, mse_test_accuracy]
})
print(loss_function_table.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.plot(range(1, NUM_EPOCHS + 1), cce_train_losses, label="Cross-Entropy Loss")
plt.plot(range(1, NUM_EPOCHS + 1), mse_train_losses, label="MSE Loss")
plt.title("Part 3: Training Loss - Cross-Entropy vs MSE")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.show()

In [ ]:
#Regression Experiment 
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

housing_data = fetch_california_housing()
X_housing, y_housing = housing_data.data, housing_data.target

X_housing_train, X_housing_test, y_housing_train, y_housing_test = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_housing_train_scaled = scaler.fit_transform(X_housing_train)
X_housing_test_scaled = scaler.transform(X_housing_test)

X_housing_train_tensor = torch.tensor(X_housing_train_scaled, dtype=torch.float32)
y_housing_train_tensor = torch.tensor(y_housing_train, dtype=torch.float32).view(-1, 1)
X_housing_test_tensor = torch.tensor(X_housing_test_scaled, dtype=torch.float32)
y_housing_test_tensor = torch.tensor(y_housing_test, dtype=torch.float32).view(-1, 1)

class RegressionMLP(nn.Module):
    def __init__(self, input_size):
        super(RegressionMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)

    def forward(self, x):
        a1 = torch.relu(self.fc1(x))
        a2 = torch.relu(self.fc2(a1))
        return self.fc3(a2)

set_seed(SEED)
regression_model = RegressionMLP(X_housing_train_scaled.shape[1])
regression_optimizer = torch.optim.Adam(regression_model.parameters(), lr=0.01)
regression_loss_fn = nn.MSELoss()

regression_epochs = 100

for epoch in range(regression_epochs):
    regression_model.train()
    regression_optimizer.zero_grad()
    predictions = regression_model(X_housing_train_tensor)
    loss = regression_loss_fn(predictions, y_housing_train_tensor)
    loss.backward()
    regression_optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Regression Epoch {epoch+1}/{regression_epochs} - Training MSE: {loss.item():.4f}")

regression_model.eval()
with torch.no_grad():
    test_predictions = regression_model(X_housing_test_tensor)
    mse = nn.MSELoss()(test_predictions, y_housing_test_tensor).item()
    rmse = np.sqrt(mse)
    mae = torch.mean(torch.abs(test_predictions - y_housing_test_tensor)).item()

print("Regression Test MSE:", mse)
print("Regression Test RMSE:", rmse)
print("Regression Test MAE:", mae)

In [ ]:
#Part 4
def get_optimizer(name, params, lr):
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr)
    elif name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    elif name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr)
    elif name == "adam":
        return torch.optim.Adam(params, lr=lr)

def train_optimizer_model(optimizer_name, learning_rate, target_accuracy=0.85):
    set_seed(SEED)
    model = ClassificationMLP().to(device)
    optimizer = get_optimizer(optimizer_name, model.parameters(), learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

    train_losses, val_accuracies = [], []
    epoch_reaching_target = "Not reached"

    start_time = time.time()

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss, num_batches = 0.0, 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1

        train_losses.append(epoch_loss / num_batches)
        _, val_accuracy = evaluate_model(model, X_val_tensor, y_val_tensor)
        val_accuracies.append(val_accuracy)

        if epoch_reaching_target == "Not reached" and val_accuracy >= target_accuracy:
            epoch_reaching_target = epoch + 1

    wall_clock_time = time.time() - start_time

    return {
        "train_losses": train_losses, "val_accuracies": val_accuracies,
        "epoch_reaching_target": epoch_reaching_target,
        "final_val_accuracy": val_accuracies[-1], "wall_clock_time": wall_clock_time
    }

In [ ]:
#Experiment A 
optimizer_names = ["sgd", "sgd_momentum", "rmsprop", "adam"]
same_lr = 0.01

experiment_a_results = {}
for name in optimizer_names:
    print("\nTraining with optimizer:", name, "- Experiment A (same LR)")
    experiment_a_results[name] = train_optimizer_model(name, same_lr)

experiment_a_rows = []
for name in optimizer_names:
    result = experiment_a_results[name]
    experiment_a_rows.append([
        name, same_lr, result["epoch_reaching_target"],
        result["final_val_accuracy"], result["wall_clock_time"]
    ])

experiment_a_table = pd.DataFrame(experiment_a_rows, columns=[
    "Optimizer", "Learning Rate", "Epochs to Reach 85% Validation Accuracy",
    "Final Validation Accuracy", "Wall-clock Time"
])
print(experiment_a_table.to_string(index=False))

In [ ]:
#Experiment B
candidate_learning_rates = {
    "sgd": [0.01, 0.05, 0.1],
    "sgd_momentum": [0.001, 0.01, 0.05],
    "rmsprop": [0.0001, 0.001, 0.01],
    "adam": [0.0001, 0.001, 0.01]
}

best_learning_rates = {}
experiment_b_results = {}

for name in optimizer_names:
    best_val_accuracy = -1
    best_lr, best_result = None, None

    for lr in candidate_learning_rates[name]:
        print("\nTuning optimizer:", name, "- Trying learning rate:", lr)
        result = train_optimizer_model(name, lr)
        if result["final_val_accuracy"] > best_val_accuracy:
            best_val_accuracy = result["final_val_accuracy"]
            best_lr = lr
            best_result = result

    best_learning_rates[name] = best_lr
    experiment_b_results[name] = best_result
    print(f"Selected learning rate for {name}: {best_lr}")

experiment_b_rows = []
for name in optimizer_names:
    result = experiment_b_results[name]
    experiment_b_rows.append([
        name, best_learning_rates[name], result["epoch_reaching_target"],
        result["final_val_accuracy"], result["wall_clock_time"]
    ])

experiment_b_table = pd.DataFrame(experiment_b_rows, columns=[
    "Optimizer", "Learning Rate", "Epochs to Reach 85% Validation Accuracy",
    "Final Validation Accuracy", "Wall-clock Time"
])
print(experiment_b_table.to_string(index=False))

plt.figure(figsize=(8, 5))
for name in optimizer_names:
    plt.plot(range(1, NUM_EPOCHS + 1), experiment_b_results[name]["train_losses"], label=name)
plt.title("Part 4: Training Loss by Optimizer (Tuned Learning Rates)")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.show()

Part 5 intentionally creates overfitting so that Part 6's regularization techniques have a real problem to solve. We do this by pairing a high-capacity model with a very small training set — the model has far more parameters than it needs, so instead of learning general patterns it starts memorizing individual training examples.

In [ ]:
#PART 5
from sklearn.model_selection import train_test_split

subset_train_size = 2000

X_overfit_subset, _, y_overfit_subset, _ = train_test_split(
    X_train, y_train,
    train_size=subset_train_size,
    random_state=SEED,
    stratify=y_train
)

X_overfit_tensor = torch.tensor(X_overfit_subset, dtype=torch.float32).to(device)
y_overfit_tensor = torch.tensor(y_overfit_subset, dtype=torch.long).to(device)

print("Training samples used (overfitting subset):", X_overfit_subset.shape[0])
print("Validation samples:", X_val.shape[0])
print("Number of classes:", len(np.unique(y_overfit_subset)))

overfit_class_counts = pd.Series(y_overfit_subset).value_counts().sort_index()
overfit_distribution_table = pd.DataFrame({
    "Class": overfit_class_counts.index,
    "Class Name": [class_names[i] for i in overfit_class_counts.index],
    "Count": overfit_class_counts.values
})
print(overfit_distribution_table.to_string(index=False))

2000 samples across 10 classes gives roughly 200 examples per class — still small enough that a high-capacity network can memorize individual images rather than learn transferable features, but large enough to train in reasonable time with mini-batches.

In [ ]:
class OverfitMLP(nn.Module):
    def __init__(self):
        super(OverfitMLP, self).__init__()
        # Four hidden layers of 512 units each, deliberately oversized
        # for a 2000-sample training set so the model can memorize it
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 512)
        self.fc4 = nn.Linear(512, 512)
        self.fc5 = nn.Linear(512, 10)

    def forward(self, x):
        a1 = torch.relu(self.fc1(x))
        a2 = torch.relu(self.fc2(a1))
        a3 = torch.relu(self.fc3(a2))
        a4 = torch.relu(self.fc4(a3))
        return self.fc5(a4)

This network has roughly 1.3 million trainable parameters for only 2000 training images — over 600 parameters per sample. No dropout, no weight decay, no early stopping, and no batch normalization: every safeguard is deliberately absent so overfitting shows up as clearly as possible.

In [ ]:
def compute_accuracy(logits, y):
    predictions = torch.argmax(logits, dim=1)
    return (predictions == y).float().mean().item()

set_seed(SEED)
overfit_model = OverfitMLP().to(device)
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.001)
overfit_loss_fn = nn.CrossEntropyLoss()

overfit_dataset = TensorDataset(X_overfit_tensor, y_overfit_tensor)
overfit_loader = DataLoader(overfit_dataset, batch_size=128, shuffle=True,
                             generator=torch.Generator().manual_seed(SEED))

overfit_epochs = 150

overfit_train_losses, overfit_val_losses = [], []
overfit_train_accuracies, overfit_val_accuracies = [], []

for epoch in range(overfit_epochs):
    overfit_model.train()
    epoch_loss, num_batches = 0.0, 0
    correct_predictions, total_samples = 0, 0

    for X_batch, y_batch in overfit_loader:
        overfit_optimizer.zero_grad()
        logits = overfit_model(X_batch)
        loss = overfit_loss_fn(logits, y_batch)
        loss.backward()
        overfit_optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1
        correct_predictions += (torch.argmax(logits, dim=1) == y_batch).sum().item()
        total_samples += y_batch.size(0)

    avg_train_loss = epoch_loss / num_batches
    train_accuracy = correct_predictions / total_samples
    val_loss, val_accuracy = evaluate_model(overfit_model, X_val_tensor, y_val_tensor)

    overfit_train_losses.append(avg_train_loss)
    overfit_val_losses.append(val_loss)
    overfit_train_accuracies.append(train_accuracy)
    overfit_val_accuracies.append(val_accuracy)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{overfit_epochs} - Train Loss: {avg_train_loss:.4f} - Train Acc: {train_accuracy:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_accuracy:.4f}")

print("\nFinal training accuracy:", overfit_train_accuracies[-1])
if overfit_train_accuracies[-1] > 0.99:
    print("Target exceeded: training accuracy is above 99% as required.")
else:
    print("Training accuracy did not exceed 99% - increase overfit_epochs and re-run this cell.")

In [ ]:
epoch_of_min_val_loss = overfit_val_losses.index(min(overfit_val_losses)) + 1

plt.figure(figsize=(9, 5))
plt.plot(range(1, overfit_epochs + 1), overfit_train_losses, label="Training Loss")
plt.plot(range(1, overfit_epochs + 1), overfit_val_losses, label="Validation Loss")
plt.axvline(x=epoch_of_min_val_loss, color="red", linestyle="--",
            label=f"Separation point (epoch {epoch_of_min_val_loss})")
plt.title("Part 5: Training vs Validation Loss (Forced Overfitting)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(range(1, overfit_epochs + 1), overfit_train_accuracies, label="Training Accuracy")
plt.plot(range(1, overfit_epochs + 1), overfit_val_accuracies, label="Validation Accuracy")
plt.axvline(x=epoch_of_min_val_loss, color="red", linestyle="--",
            label=f"Separation point (epoch {epoch_of_min_val_loss})")
plt.title("Part 5: Training vs Validation Accuracy (Forced Overfitting)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
final_train_accuracy = overfit_train_accuracies[-1]
final_val_accuracy = overfit_val_accuracies[-1]
generalization_gap = final_train_accuracy - final_val_accuracy

print("Final training accuracy:", final_train_accuracy)
print("Final validation accuracy:", final_val_accuracy)
print("Generalization gap (train acc - val acc):", generalization_gap)
print("Epoch of minimum validation loss (separation point):", epoch_of_min_val_loss)
print("Minimum validation loss:", min(overfit_val_losses))

In [ ]:
#part6
import torchvision.transforms as T

def compute_epoch_metrics(model, loader, loss_fn, l1_lambda=0.0):
    model.eval()
    with torch.no_grad():
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            total_loss += loss.item() * y_batch.size(0)
            correct += (torch.argmax(logits, dim=1) == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / total, correct / total

def train_regularized_overfit_model(model, optimizer, X_train_t, y_train_t,
                                     num_epochs=150, batch_size=128,
                                     l1_lambda=0.0, use_augmentation=False,
                                     use_early_stopping=False, patience=10, min_delta=0.0):
    loss_fn = nn.CrossEntropyLoss()
    train_dataset_local = TensorDataset(X_train_t, y_train_t)
    train_loader_local = DataLoader(train_dataset_local, batch_size=batch_size, shuffle=True,
                                     generator=torch.Generator().manual_seed(SEED))
    val_loader_local = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=512)

    augmentation_transform = T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10)
    ])

    def augment_batch(X_batch):
        images = X_batch.view(-1, 1, 28, 28)
        augmented = torch.stack([augmentation_transform(img) for img in images])
        return augmented.view(-1, 784)

    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    best_val_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    best_model_state = None
    stopped_epoch = num_epochs

    for epoch in range(num_epochs):
        model.train()
        for X_batch, y_batch in train_loader_local:
            if use_augmentation:
                X_batch = augment_batch(X_batch)

            optimizer.zero_grad()
            logits = model(X_batch)
            classification_loss = loss_fn(logits, y_batch)

            if l1_lambda > 0:
                l1_penalty = sum(p.abs().sum() for name, p in model.named_parameters() if "weight" in name)
                total_loss = classification_loss + l1_lambda * l1_penalty
            else:
                total_loss = classification_loss

            total_loss.backward()
            optimizer.step()

        train_loss, train_acc = compute_epoch_metrics(model, train_loader_local, loss_fn)
        val_loss, val_acc = compute_epoch_metrics(model, val_loader_local, loss_fn)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1

        if use_early_stopping and epochs_without_improvement >= patience:
            stopped_epoch = epoch + 1
            break

    if use_early_stopping and best_model_state is not None:
        model.load_state_dict(best_model_state)
        final_train_loss, final_train_acc = compute_epoch_metrics(model, train_loader_local, loss_fn)
        final_val_loss, final_val_acc = compute_epoch_metrics(model, val_loader_local, loss_fn)
    else:
        final_train_acc, final_val_acc = train_accs[-1], val_accs[-1]

    return {
        "train_losses": train_losses, "val_losses": val_losses,
        "train_accs": train_accs, "val_accs": val_accs,
        "final_train_acc": final_train_acc, "final_val_acc": final_val_acc,
        "best_epoch": best_epoch, "stopped_epoch": stopped_epoch
    }

In [ ]:
class OverfitMLPDropout(nn.Module):
    def __init__(self, dropout_prob):
        super(OverfitMLPDropout, self).__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 512)
        self.fc4 = nn.Linear(512, 512)
        self.fc5 = nn.Linear(512, 10)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        a1 = self.dropout(torch.relu(self.fc1(x)))
        a2 = self.dropout(torch.relu(self.fc2(a1)))
        a3 = self.dropout(torch.relu(self.fc3(a2)))
        a4 = self.dropout(torch.relu(self.fc4(a3)))
        return self.fc5(a4)

class OverfitMLPBatchNorm(nn.Module):
    def __init__(self):
        super(OverfitMLPBatchNorm, self).__init__()
        self.fc1 = nn.Linear(784, 512); self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 512); self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = nn.Linear(512, 512); self.bn3 = nn.BatchNorm1d(512)
        self.fc4 = nn.Linear(512, 512); self.bn4 = nn.BatchNorm1d(512)
        self.fc5 = nn.Linear(512, 10)

    def forward(self, x):
        a1 = torch.relu(self.bn1(self.fc1(x)))
        a2 = torch.relu(self.bn2(self.fc2(a1)))
        a3 = torch.relu(self.bn3(self.fc3(a2)))
        a4 = torch.relu(self.bn4(self.fc4(a3)))
        return self.fc5(a4)

All regularization variants reuse the exact 784→512→512→512→512→10 architecture from Part 5 wherever the method doesn't structurally require otherwise (Dropout and BatchNorm need extra layers inserted, so they get their own class, but the layer sizes are unchanged). Everything trains on the same 2000-sample subset with the same seed, batch size, and epoch count except where the method itself is defined by changing one of those (Early Stopping, More Data).

In [ ]:
l2_lambda_values = [1e-4, 1e-3, 1e-2]
l2_results_by_lambda = {}

for lam in l2_lambda_values:
    set_seed(SEED)
    model = OverfitMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=lam)
    result = train_regularized_overfit_model(model, optimizer, X_overfit_tensor, y_overfit_tensor, num_epochs=150)
    l2_results_by_lambda[lam] = result
    print(f"L2 lambda={lam} - Final Train Acc: {result['final_train_acc']:.4f} - Final Val Acc: {result['final_val_acc']:.4f} - Gap: {result['final_train_acc']-result['final_val_acc']:.4f}")

Testing three lambda values (weak, medium, strong) lets us see the regularization-strength tradeoff directly: too weak and the gap barely closes; too strong and training accuracy itself starts to suffer because the penalty is now fighting the model's ability to fit even its own training data.

In [ ]:
set_seed(SEED)
l1_model = OverfitMLP().to(device)
l1_optimizer = torch.optim.Adam(l1_model.parameters(), lr=0.001)
l1_lambda_value = 1e-5
l1_result = train_regularized_overfit_model(l1_model, l1_optimizer, X_overfit_tensor, y_overfit_tensor,
                                             num_epochs=150, l1_lambda=l1_lambda_value)

all_weights = torch.cat([p.flatten() for name, p in l1_model.named_parameters() if "weight" in name])
percent_near_zero = (all_weights.abs() < 1e-3).float().mean().item() * 100

print("L1 - Final Train Acc:", l1_result["final_train_acc"])
print("L1 - Final Val Acc:", l1_result["final_val_acc"])
print("L1 - Gap:", l1_result["final_train_acc"] - l1_result["final_val_acc"])
print("Percentage of weights below 1e-3:", percent_near_zero)

L1's defining behavior is pushing many weights all the way toward zero rather than just shrinking them uniformly (which is what L2 does). The sparsity percentage tells you how many connections became effectively "pruned" — a high percentage means the network found it could discard many weights entirely and still fit the data, evidence the original network was over-parameterized for this task.

In [ ]:
dropout_rate_values = [0.2, 0.3, 0.5]
dropout_results_by_rate = {}

for rate in dropout_rate_values:
    set_seed(SEED)
    model = OverfitMLPDropout(rate).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    result = train_regularized_overfit_model(model, optimizer, X_overfit_tensor, y_overfit_tensor, num_epochs=150)
    dropout_results_by_rate[rate] = result
    print(f"Dropout p={rate} - Final Train Acc: {result['final_train_acc']:.4f} - Final Val Acc: {result['final_val_acc']:.4f} - Gap: {result['final_train_acc']-result['final_val_acc']:.4f}")

In [ ]:
set_seed(SEED)
batchnorm_model = OverfitMLPBatchNorm().to(device)
batchnorm_optimizer = torch.optim.Adam(batchnorm_model.parameters(), lr=0.001)
batchnorm_result = train_regularized_overfit_model(batchnorm_model, batchnorm_optimizer, X_overfit_tensor, y_overfit_tensor, num_epochs=150)

print("BatchNorm - Final Train Acc:", batchnorm_result["final_train_acc"])
print("BatchNorm - Final Val Acc:", batchnorm_result["final_val_acc"])
print("BatchNorm - Gap:", batchnorm_result["final_train_acc"] - batchnorm_result["final_val_acc"])

In [ ]:
set_seed(SEED)
earlystop_model = OverfitMLP().to(device)
earlystop_optimizer = torch.optim.Adam(earlystop_model.parameters(), lr=0.001)
early_stopping_patience = 10
earlystop_result = train_regularized_overfit_model(
    earlystop_model, earlystop_optimizer, X_overfit_tensor, y_overfit_tensor,
    num_epochs=150, use_early_stopping=True, patience=early_stopping_patience
)

print("Early Stopping - Patience:", early_stopping_patience)
print("Early Stopping - Stopped at epoch:", earlystop_result["stopped_epoch"])
print("Early Stopping - Best epoch:", earlystop_result["best_epoch"])
print("Early Stopping - Final Train Acc:", earlystop_result["final_train_acc"])
print("Early Stopping - Final Val Acc:", earlystop_result["final_val_acc"])
print("Early Stopping - Gap:", earlystop_result["final_train_acc"] - earlystop_result["final_val_acc"])

In [ ]:
set_seed(SEED)
augment_model = OverfitMLP().to(device)
augment_optimizer = torch.optim.Adam(augment_model.parameters(), lr=0.001)
augment_result = train_regularized_overfit_model(
    augment_model, augment_optimizer, X_overfit_tensor, y_overfit_tensor,
    num_epochs=150, use_augmentation=True
)

print("Augmentation - Final Train Acc:", augment_result["final_train_acc"])
print("Augmentation - Final Val Acc:", augment_result["final_val_acc"])
print("Augmentation - Gap:", augment_result["final_train_acc"] - augment_result["final_val_acc"])

In [ ]:
more_data_sizes = [10000, 20000]
more_data_results = {}

for size in more_data_sizes:
    X_more_data, _, y_more_data, _ = train_test_split(
        X_train, y_train, train_size=size, random_state=SEED, stratify=y_train
    )
    X_more_data_tensor = torch.tensor(X_more_data, dtype=torch.float32).to(device)
    y_more_data_tensor = torch.tensor(y_more_data, dtype=torch.long).to(device)

    set_seed(SEED)
    model = OverfitMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    result = train_regularized_overfit_model(model, optimizer, X_more_data_tensor, y_more_data_tensor, num_epochs=100)
    more_data_results[size] = result
    print(f"More Data (n={size}) - Final Train Acc: {result['final_train_acc']:.4f} - Final Val Acc: {result['final_val_acc']:.4f} - Gap: {result['final_train_acc']-result['final_val_acc']:.4f}")

In [ ]:
summary_rows = [
    ["Baseline (no regularization)", "-", final_train_accuracy, final_val_accuracy, generalization_gap],
]

for lam in l2_lambda_values:
    r = l2_results_by_lambda[lam]
    summary_rows.append(["L2 weight decay", f"lambda={lam}", r["final_train_acc"], r["final_val_acc"], r["final_train_acc"] - r["final_val_acc"]])

summary_rows.append(["L1 penalty", f"lambda={l1_lambda_value}", l1_result["final_train_acc"], l1_result["final_val_acc"], l1_result["final_train_acc"] - l1_result["final_val_acc"]])

for rate in dropout_rate_values:
    r = dropout_results_by_rate[rate]
    summary_rows.append(["Dropout", f"p={rate}", r["final_train_acc"], r["final_val_acc"], r["final_train_acc"] - r["final_val_acc"]])

summary_rows.append(["Batch Normalization", "-", batchnorm_result["final_train_acc"], batchnorm_result["final_val_acc"], batchnorm_result["final_train_acc"] - batchnorm_result["final_val_acc"]])
summary_rows.append(["Early Stopping", f"patience={early_stopping_patience}", earlystop_result["final_train_acc"], earlystop_result["final_val_acc"], earlystop_result["final_train_acc"] - earlystop_result["final_val_acc"]])
summary_rows.append(["Data Augmentation", "flip + rotation", augment_result["final_train_acc"], augment_result["final_val_acc"], augment_result["final_train_acc"] - augment_result["final_val_acc"]])

for size in more_data_sizes:
    r = more_data_results[size]
    summary_rows.append(["More Training Data", f"n={size}", r["final_train_acc"], r["final_val_acc"], r["final_train_acc"] - r["final_val_acc"]])

part6_summary_table = pd.DataFrame(summary_rows, columns=["Method", "Setting", "Train Accuracy", "Validation Accuracy", "Gap"])
print(part6_summary_table.to_markdown(index=False))

In [ ]:
l2_gaps = [l2_results_by_lambda[lam]["final_train_acc"] - l2_results_by_lambda[lam]["final_val_acc"] for lam in l2_lambda_values]

plt.figure(figsize=(7, 5))
plt.plot(l2_lambda_values, l2_gaps, marker="o", label="Generalization Gap")
plt.xscale("log")
plt.title("Part 6.10: Generalization Gap vs L2 Lambda")
plt.xlabel("L2 Lambda (log scale)")
plt.ylabel("Generalization Gap (Train Acc - Val Acc)")
plt.legend()
plt.show()

dropout_gaps = [dropout_results_by_rate[rate]["final_train_acc"] - dropout_results_by_rate[rate]["final_val_acc"] for rate in dropout_rate_values]

plt.figure(figsize=(7, 5))
plt.plot(dropout_rate_values, dropout_gaps, marker="o", label="Generalization Gap")
plt.title("Part 6.10: Generalization Gap vs Dropout Rate")
plt.xlabel("Dropout Probability")
plt.ylabel("Generalization Gap (Train Acc - Val Acc)")
plt.legend()
plt.show()

Among all methods tested, More Training Data (n=20000) produced the strongest result: validation accuracy rose to 88.2%, the highest of any configuration, while training accuracy stayed at 99.6% — meaning the gap closed without sacrificing the model's fitting ability. Early Stopping achieved the numerically smallest gap (0.095), but only by halting training at epoch 16, which also reduced training accuracy to 91.7% — closing the gap by holding the model back rather than improving generalization. L2 and Dropout gave smaller, less consistent improvements: their best settings (lambda=0.0001 and p=0.2 respectively) improved validation accuracy modestly but left most of the original gap intact. Overall, more data gave the largest genuine gap reduction for the smallest cost to training accuracy.

In [ ]:
#part 7
search_space = {
    "learning_rate": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "hidden_size": [64, 128, 256],
    "dropout": [0.0, 0.1, 0.2, 0.3, 0.5]
}

class SearchMLP(nn.Module):
    def __init__(self, hidden_size, dropout_prob):
        super(SearchMLP, self).__init__()
        self.fc1 = nn.Linear(784, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, 10)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        a1 = self.dropout(torch.relu(self.fc1(x)))
        a2 = self.dropout(torch.relu(self.fc2(a1)))
        return self.fc3(a2)

random.seed(SEED)
num_configurations = 12
sampled_configs = []
for _ in range(num_configurations):
    config = {
        "learning_rate": random.choice(search_space["learning_rate"]),
        "hidden_size": random.choice(search_space["hidden_size"]),
        "dropout": random.choice(search_space["dropout"])
    }
    sampled_configs.append(config)

print("Search space:")
for key, values in search_space.items():
    print(f"  {key}: {values}")
print("\nSampled configurations:")
for i, config in enumerate(sampled_configs):
    print(f"Config {i+1}: {config}")

The search space covers three hyperparameters: learning rate, hidden layer width, and dropout rate. Dropout is included here deliberately — Part 6 found dropout (p=0.2) was the strongest true regularization technique tested, so rather than fixing it at exactly 0.2, we let random search re-tune it specifically for the full-size training set, which behaves differently from the small 2000-sample subset used in Part 6.

In [ ]:
def cross_validate_config(config, num_folds=5, num_epochs=8, batch_size=128):
    kfold = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []

    for fold_index, (train_idx, val_idx) in enumerate(kfold.split(X_full_train, y_full_train)):
        set_seed(SEED)
        model = SearchMLP(config["hidden_size"], config["dropout"]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
        loss_fn = nn.CrossEntropyLoss()

        X_fold_train = torch.tensor(X_full_train[train_idx], dtype=torch.float32).to(device)
        y_fold_train = torch.tensor(y_full_train[train_idx], dtype=torch.long).to(device)
        X_fold_val = torch.tensor(X_full_train[val_idx], dtype=torch.float32).to(device)
        y_fold_val = torch.tensor(y_full_train[val_idx], dtype=torch.long).to(device)

        fold_loader = DataLoader(TensorDataset(X_fold_train, y_fold_train), batch_size=batch_size, shuffle=True,
                                  generator=torch.Generator().manual_seed(SEED))

        for epoch in range(num_epochs):
            model.train()
            for X_batch, y_batch in fold_loader:
                optimizer.zero_grad()
                logits = model(X_batch)
                loss = loss_fn(logits, y_batch)
                loss.backward()
                optimizer.step()

        _, fold_val_accuracy = evaluate_model(model, X_fold_val, y_fold_val)
        fold_scores.append(fold_val_accuracy)

    return fold_scores

In [ ]:
search_results = []

for run_index, config in enumerate(sampled_configs):
    print(f"\nRunning config {run_index+1}/{num_configurations}: {config}")
    fold_scores = cross_validate_config(config)
    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)

    search_results.append({
        "Run": run_index + 1,
        "Learning Rate": config["learning_rate"],
        "Hidden Size": config["hidden_size"],
        "Dropout": config["dropout"],
        "Fold Scores": fold_scores,
        "Mean CV Accuracy": mean_score,
        "Std CV Accuracy": std_score
    })

    print(f"Fold scores: {[round(s, 4) for s in fold_scores]}")
    print(f"Mean: {mean_score:.4f} - Std: {std_score:.4f}")

In [ ]:
search_results_table = pd.DataFrame(search_results)
search_results_table = search_results_table.sort_values("Mean CV Accuracy", ascending=False).reset_index(drop=True)

top_5_table = search_results_table[["Run", "Learning Rate", "Hidden Size", "Dropout", "Mean CV Accuracy", "Std CV Accuracy"]].head(5)
print("Top 5 configurations by mean cross-validation accuracy:")
print(top_5_table.to_markdown(index=False))

best_config_row = search_results_table.iloc[0]
print("\nSelected configuration:")
print(best_config_row[["Learning Rate", "Hidden Size", "Dropout", "Mean CV Accuracy", "Std CV Accuracy"]])

In [ ]:
best_learning_rate = float(best_config_row["Learning Rate"])
best_hidden_size = int(best_config_row["Hidden Size"])
best_dropout = float(best_config_row["Dropout"])

set_seed(SEED)
final_search_model = SearchMLP(best_hidden_size, best_dropout).to(device)
final_search_optimizer = torch.optim.Adam(final_search_model.parameters(), lr=best_learning_rate)
final_search_loss_fn = nn.CrossEntropyLoss()

final_search_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,
                                  generator=torch.Generator().manual_seed(SEED))

final_search_epochs = 15
for epoch in range(final_search_epochs):
    final_search_model.train()
    for X_batch, y_batch in final_search_loader:
        final_search_optimizer.zero_grad()
        logits = final_search_model(X_batch)
        loss = final_search_loss_fn(logits, y_batch)
        loss.backward()
        final_search_optimizer.step()
    _, val_acc = evaluate_model(final_search_model, X_val_tensor, y_val_tensor)
    print(f"Epoch {epoch+1}/{final_search_epochs} - Val Accuracy: {val_acc:.4f}")

final_search_val_loss, final_search_val_accuracy = evaluate_model(final_search_model, X_val_tensor, y_val_tensor)
print("\nFinal validation accuracy:", final_search_val_accuracy)

In [ ]:
final_search_model.eval()
with torch.no_grad():
    test_logits = final_search_model(X_test_tensor)
    test_loss = nn.CrossEntropyLoss()(test_logits, y_test_tensor).item()
    test_predictions = torch.argmax(test_logits, dim=1).cpu().numpy()

y_test_np = y_test_tensor.cpu().numpy()

final_test_accuracy = (test_predictions == y_test_np).mean()

from sklearn.metrics import precision_recall_fscore_support

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test_np, test_predictions, average="macro"
)

print("Final Test Loss:", test_loss)
print("Final Test Accuracy:", final_test_accuracy)
print("Macro Precision:", macro_precision)
print("Macro Recall:", macro_recall)
print("Macro F1:", macro_f1)

improvement_pp = (final_test_accuracy - baseline_test_accuracy) * 100
print("\nPart 2 Baseline Test Accuracy:", baseline_test_accuracy)
print("Improvement over Part 2 baseline (percentage points):", improvement_pp)

In [ ]:
class_label_names = [class_names[i] for i in range(10)]
conf_matrix = confusion_matrix(y_test_np, test_predictions)

plt.figure(figsize=(9, 8))
plt.imshow(conf_matrix, cmap="Blues")
plt.title("Part 7: Confusion Matrix - Final Tuned Model on Test Set")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(range(10), class_label_names, rotation=45, ha="right")
plt.yticks(range(10), class_label_names)
plt.colorbar()
for i in range(10):
    for j in range(10):
        plt.text(j, i, conf_matrix[i, j], ha="center", va="center",
                  color="white" if conf_matrix[i, j] > conf_matrix.max() / 2 else "black")
plt.tight_layout()
plt.show()

In [ ]:
final_comparison_rows = [
    ["Part 2 Baseline", "784-128-64-10, ReLU, SGD, lr=0.01", None, round(baseline_test_accuracy, 4), "No regularization"],
    ["Part 5 Overfitted", "784-512-512-512-512-10, 2000 samples, Adam", round(final_val_accuracy, 4), None, "Deliberately overfit (train acc 99.3%), not tested"],
    ["Part 6 Best (Dropout p=0.2)", "Same arch, dropout=0.2", 0.845167, None, "Best true regularization technique on 2000-sample subset"],
    ["Part 7 Final (CV-Tuned Random Search)", "784-128-64-10, Adam, lr=0.001, dropout=0.1", round(final_search_val_accuracy, 4), round(final_test_accuracy, 4), "Selected via 5-fold CV over 12 configs, retrained on full training split"],
]

final_comparison_table = pd.DataFrame(
    final_comparison_rows,
    columns=["Model", "Main Configuration", "Validation Accuracy", "Test Accuracy", "Notes"]
)
print(final_comparison_table.to_markdown(index=False))